In [1]:
!pip install -q transformers accelerate torch

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"Model device: {model.device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Vocab size: {tokenizer.vocab_size}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model device: cuda:0
Model parameters: 1,100,048,384
Vocab size: 32000


In [2]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [3]:
from transformers import pipeline

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
output = generator(
    "The future of artificial intelligence is",
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_p = 0.9
    )
print(output[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The future of artificial intelligence is here: AI has been developed to the point where it can perform tasks previously thought to be beyond human control. As AI continues to evolve, we can expect it to have a significant impact on the world of work. Some of the ways in which AI is already impacting the job market include:

1. Automation: As AI continues to improve, it is expected to become more automated. This means that more jobs will be automated, freeing up workers


In [5]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain what a transformer is in 3 sentences."},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("Fomatted prompt:")
print(repr(prompt))

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.1
    )
new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)
print("\nResponse:", response)

Fomatted prompt:
'<|system|>\nYou are a helpful assistant.</s>\n<|user|>\nExplain what a transformer is in 3 sentences.</s>\n<|assistant|>\n'

Response: A transformer is a device that converts one power source (such as electricity) into another, typically with higher voltage and current. It is named after the Greek word "transformer," which means "to change." Transformers are used in various applications such as power distribution, telecommunications, and energy storage. They are designed to handle large amounts of electricity and can be used for both AC and DC power sources. The process of converting one power source into another involves changing the electrical resistance, voltage, and current. A transformer is essential for maintaining consistent power quality and reliability in power systems.


In [7]:
# 提取 RoPE 的频率参数
rope = model.model.rotary_emb

# 查看 inv_freq（θ_i = 1/10000^(2i/d)）
print("RoPE inv_freq shape:", rope.inv_freq.shape)
print("First few freqs:", rope.inv_freq[:5])

# 验证：这些频率和你上午学的公式一致
import torch
d_head = 64  # 2048/32 = 64
expected = 1.0 / (10000 ** (torch.arange(0, d_head, 2).float() / d_head))
print("Expected freqs:", expected[:5])


RoPE inv_freq shape: torch.Size([32])
First few freqs: tensor([1.0000, 0.7499, 0.5623, 0.4217, 0.3162], device='cuda:0')
Expected freqs: tensor([1.0000, 0.7499, 0.5623, 0.4217, 0.3162])
